# LLM Model Taxonomy

**Course:** [Natural Language Processing](https://ml-viz-ruby.vercel.app/courses/nlp/04-llm-model-taxonomy)

This notebook draws the three attention-mask shapes that distinguish encoder-only, decoder-only, and encoder-decoder Transformers, runs tiny NumPy forward passes for an encoder-style pooler and a decoder-style next-token head, and visualises representative model sizes on a log scale.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## 1. Three attention-mask shapes

The defining structural difference between the three Transformer families is *which entries of the $L \times L$ attention score matrix are allowed to be non-zero*. We'll build the three masks by hand and plot them side by side.

- **Encoder (bidirectional):** every position attends to every position.
- **Decoder (causal):** position $i$ attends only to $j \le i$ — a strictly lower-triangular pattern.
- **Encoder-decoder cross-attention:** decoder queries (rows) attend to *all* encoder positions (columns). The shape is generally rectangular ($L_{\text{tgt}} \times L_{\text{src}}$) and fully populated.

In all cases we represent 'allowed' as $0$ and 'forbidden' as $-\infty$ — the additive form you'd add to the pre-softmax logits.

In [ ]:
def bidirectional_mask(L):
    """Encoder mask: every position can attend to every other position."""
    return np.zeros((L, L))

def causal_mask(L):
    """Decoder mask: strictly lower-triangular; future positions are -inf."""
    M = np.zeros((L, L))
    M[np.triu_indices(L, k=1)] = -np.inf
    return M

def cross_mask(L_tgt, L_src):
    """Encoder-decoder cross-attention: every target query sees every source key."""
    return np.zeros((L_tgt, L_src))

L = 8
masks = [
    ("Encoder (bidirectional)",      bidirectional_mask(L)),
    ("Decoder (causal)",             causal_mask(L)),
    ("Encoder-Decoder (cross-attn)", cross_mask(L, L)),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, M) in zip(axes, masks):
    # Convert -inf → 0 (forbidden) and 0 → 1 (allowed) for the heatmap
    allowed = (np.isfinite(M)).astype(float)
    ax.imshow(allowed, cmap='viridis', vmin=0, vmax=1, aspect='equal')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Key position', fontsize=10)
    ax.set_ylabel('Query position', fontsize=10)
    ax.set_xticks(range(L))
    ax.set_yticks(range(L))
plt.suptitle('Attention mask shapes for the three Transformer families', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

Notice the diagonal stripe on the decoder mask: row $i$ has exactly $i+1$ allowed columns. That is *the* invariant that makes next-token prediction a valid pretraining objective — without it the model could trivially read $x_t$ when predicting $x_t$.

## 2. Toy encoder forward: bidirectional pooling → classification logit

We'll run a single attention layer over a sequence of token embeddings using the *bidirectional* mask, mean-pool the resulting representations into a sentence vector, and project to a binary classification logit. This is the inference pattern an encoder-only model like BERT uses for sentiment classification.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask):
    """Single-head scaled dot-product attention with an additive mask."""
    d = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d) + mask
    attn = softmax(scores, axis=-1)
    return attn @ V, attn

# Toy sequence of token embeddings: "this movie is great"
rng = np.random.default_rng(0)
L_enc, d_model = 4, 8
X_enc = rng.standard_normal((L_enc, d_model)) * 0.5

# Project to Q, K, V with random weights (no training — just illustrating shapes)
Wq = rng.standard_normal((d_model, d_model)) * 0.3
Wk = rng.standard_normal((d_model, d_model)) * 0.3
Wv = rng.standard_normal((d_model, d_model)) * 0.3

Q = X_enc @ Wq
K = X_enc @ Wk
V = X_enc @ Wv

H_enc, attn_enc = attention(Q, K, V, bidirectional_mask(L_enc))
print(f"Encoder output shape: {H_enc.shape}  # (L, d_model)")

# Sentence vector: mean-pool over positions
sentence_vec = H_enc.mean(axis=0)

# Tiny classification head: linear projection to a logit
W_cls = rng.standard_normal(d_model) * 0.5
logit = sentence_vec @ W_cls
prob_positive = 1 / (1 + np.exp(-logit))
print(f"Sentence vector shape: {sentence_vec.shape}")
print(f"Classification logit:  {logit:+.3f}")
print(f"P(positive):           {prob_positive:.3f}  (random weights — this is illustrative)")

Key shape facts: the encoder maps $L$ tokens to $L$ contextual vectors of size $d_{\text{model}}$, and the classification head consumes a *single* sentence vector. Notice that token 0's representation depends on tokens 1–3 too — that's bidirectionality at work.

## 3. Toy decoder forward: causal attention → next-token logits

Same building blocks; we just swap the mask and tap a different output. The decoder produces one *next-token* distribution per position. During training, position $t$'s prediction is supervised by the true token $x_{t+1}$ — next-token cross-entropy.

In [ ]:
L_dec, V_vocab = 5, 12
X_dec = rng.standard_normal((L_dec, d_model)) * 0.5

Q_d = X_dec @ Wq
K_d = X_dec @ Wk
V_d = X_dec @ Wv

H_dec, attn_dec = attention(Q_d, K_d, V_d, causal_mask(L_dec))
print(f"Decoder output shape: {H_dec.shape}")

# Verify the causal property: each row of the attention matrix should sum to 1
# and have non-zero entries only at columns <= row index.
print("\nAttention weights at each decoder position (only past + self should be non-zero):")
for i in range(L_dec):
    nonzero_cols = np.where(attn_dec[i] > 1e-6)[0].tolist()
    print(f"  position {i}: attends to {nonzero_cols}")

# Tiny LM head: project each position to vocab logits
W_lm = rng.standard_normal((d_model, V_vocab)) * 0.4
logits = H_dec @ W_lm                            # (L, V)
next_token_dists = softmax(logits, axis=-1)

print(f"\nNext-token logit matrix shape: {logits.shape}  # (L, V)")
print(f"Most-likely next token at each position: {next_token_dists.argmax(axis=-1).tolist()}")

Observe that position 0 attends only to itself, position 1 to {0, 1}, and so on — the causal mask is doing its job. The LM head produces *L* independent next-token distributions in a single forward pass; that is what makes parallel teacher-forced training so cheap.

## 4. Representative models: size, year, and use

A small reference table and a log-scale bar chart of parameter counts. The takeaway is *not* that bigger is uniformly better — see the lesson for why training tokens matter just as much. But the four orders of magnitude between BERT-base and frontier models is worth seeing.

In [ ]:
models = [
    # (name,              family,            params (B), year, primary use)
    ('BERT-base',        'encoder-only',     0.110, 2018, 'classification, NER'),
    ('BERT-large',       'encoder-only',     0.340, 2018, 'classification, NER'),
    ('RoBERTa-large',    'encoder-only',     0.355, 2019, 'classification, embeddings'),
    ('ModernBERT-base',  'encoder-only',     0.150, 2024, 'embeddings, retrieval'),
    ('T5-base',          'encoder-decoder',  0.220, 2019, 'text→text, translation'),
    ('T5-11B',           'encoder-decoder', 11.000, 2019, 'text→text, translation'),
    ('BART-large',       'encoder-decoder',  0.400, 2019, 'summarisation'),
    ('GPT-2',            'decoder-only',     1.500, 2019, 'generation'),
    ('GPT-3',            'decoder-only',   175.000, 2020, 'few-shot chat'),
    ('Llama-3-8B',       'decoder-only',     8.000, 2024, 'chat, code'),
    ('Llama-3-70B',      'decoder-only',    70.000, 2024, 'chat, code, agents'),
    ('Llama-3-405B',     'decoder-only',   405.000, 2024, 'frontier chat'),
]

print(f"{'Model':<18}{'Family':<18}{'Params (B)':>12}  {'Year':>5}  Primary use")
print('-' * 88)
for name, fam, p, year, use in models:
    print(f"{name:<18}{fam:<18}{p:>12.3f}  {year:>5}  {use}")

family_color = {
    'encoder-only':    '#2dd4bf',
    'decoder-only':    '#6366f1',
    'encoder-decoder': '#f97316',
}

fig, ax = plt.subplots(figsize=(10, 5))
names  = [m[0] for m in models]
params = [m[2] for m in models]
colors = [family_color[m[1]] for m in models]
ax.barh(names, params, color=colors, edgecolor='#0f1117')
ax.set_xscale('log')
ax.set_xlabel('Parameters (billions, log scale)', fontsize=11)
ax.set_title('Representative LLMs by parameter count and family', fontsize=11)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Legend
from matplotlib.patches import Patch
handles = [Patch(color=c, label=f) for f, c in family_color.items()]
ax.legend(handles=handles, facecolor='#1a1d27', edgecolor='#2a2d3a', loc='lower right')
plt.tight_layout()
plt.show()

Four orders of magnitude separate BERT-base from Llama-3-405B — but the encoder-only family did not get there because it didn't need to. A 150 M-parameter ModernBERT is *better* at embeddings than any of the giant decoder models, at a tiny fraction of the inference cost. 'Large' is task-specific.

## ✏️ Your turn

### Exercise: implement `causal_mask(L)`

Implement the additive causal mask used by every decoder-only Transformer. Your function should return an $L \times L$ array where every entry on or below the diagonal is $0$ and every entry strictly above the diagonal is $-\infty$. This is the single trick that makes next-token prediction a valid pretraining objective.

In [ ]:
def causal_mask_student(L):
    """
    Return an L x L additive attention mask.
    
    M[i, j] = 0      if j <= i   (allowed: past or self)
    M[i, j] = -inf   if j >  i   (forbidden: future)
    
    The mask is *added* to the pre-softmax logits, so -inf at a position
    forces softmax to put zero weight there.
    """
    # TODO(you): build and return the L x L mask described above
    pass

# Quick visual check
M = causal_mask_student(5)
print(M)

In [ ]:
# Tests
L = 5
M = causal_mask_student(L)
assert M is not None, "Should return an array, not None"
M = np.asarray(M)
assert M.shape == (L, L), f"Expected shape ({L}, {L}), got {M.shape}"

# Reference mask
ref = causal_mask(L)

# Lower triangle (including diagonal) must be 0
for i in range(L):
    for j in range(i + 1):
        assert M[i, j] == 0, f"M[{i},{j}] should be 0, got {M[i, j]}"

# Strictly upper triangle must be -inf
for i in range(L):
    for j in range(i + 1, L):
        assert M[i, j] == -np.inf, f"M[{i},{j}] should be -inf, got {M[i, j]}"

# Sanity: softmax of (zeros + mask) gives a lower-triangular weights matrix
scores = np.zeros((L, L)) + M
attn = softmax(scores, axis=-1)
for i in range(L):
    # Past + self should sum to 1; future should be 0
    assert abs(attn[i, : i + 1].sum() - 1.0) < 1e-9
    if i + 1 < L:
        assert attn[i, i + 1 :].sum() < 1e-9

print("✅ Exercise passed")

<details>
<summary>💡 Show solution</summary>

```python
def causal_mask_student(L):
    M = np.zeros((L, L))
    M[np.triu_indices(L, k=1)] = -np.inf
    return M
```

Equivalently:

```python
def causal_mask_student(L):
    M = np.full((L, L), -np.inf)
    M[np.tril_indices(L)] = 0.0
    return M
```

Both produce the same mask: zeros on and below the diagonal, $-\infty$ above. Adding $M$ to the pre-softmax scores and then applying softmax sends every future-position weight to exactly zero.
</details>